In [15]:
import pandas as pd
import glob
import os
import re

In [16]:
## rename bars to every 5 minutes

bar_dict = {
    '0-5': 'bar_0',
    '6-10': 'bar_1',
    '11-15': 'bar_2',
    '16-20': 'bar_3',
    '21-25': 'bar_4',
    '26-30': 'bar_5',
    '31-35': 'bar_6',
    '36-40': 'bar_7',
    '41-45': 'bar_8',
    '46-50': 'bar_2_0',
    '51-55': 'bar_2_1',
    '56-60': 'bar_2_2',
    '61-65': 'bar_2_3',
    '66-70': 'bar_2_4',
    '71-75': 'bar_2_5',
    '76-80': 'bar_2_6',
    '81-85': 'bar_2_7',
    '86-90': 'bar_2_8',
}

bar_dict_switched = {v: k for k, v in bar_dict.items()}


In [17]:
def clean_teams(df):
    df = df.copy()
    df = df.drop(columns=['home_advantage', 'away_advantage'])
    df['stat'] = df['category'].astype(str) + '_' + df['stat_name'].astype(str)
    df = df.drop(columns=['category', 'stat_name'])
    df['tip_id'] = df['tip_id'].astype(str).str.strip() 
    df['tip_id'] = df['tip_id'].replace(bar_dict_switched)
    mask = df["tip_id"].astype(str).str.match(r"^\d{1,2}-\d{1,2}$", na=False)
    h_pct = df.loc[mask, "home_possession"].astype(str).str.extract(r"(\d+(?:\.\d+)?)")[0].astype(float)
    a_pct = df.loc[mask, "away_possession"].astype(str).str.extract(r"(\d+(?:\.\d+)?)")[0].astype(float)
    df.loc[mask, "home_value"] = h_pct.values 
    df.loc[mask, "away_value"] = a_pct.values
    df.loc[mask, "stat"] = "possession_" + df.loc[mask, "tip_id"].str.replace("-", "_", regex=False)
    df = df.drop(columns=['tip_id', 'home_possession', 'away_possession'])
    df = df[['stat', 'home_value', 'away_value', 'match_id', 'date']]
    return df

In [19]:
from datetime import datetime, timedelta
import os
import re
import pandas as pd
import glob
from datetime import datetime

dfs = []


folder = glob.glob('G:/My Drive/GitHubProjects/MLS/data/raw/matches/teams/*.csv')

for file in folder:
    if re.search(r"\\stats_.*\.csv", file):
        modified = datetime.fromtimestamp(os.path.getmtime(file))
        if modified.year == 2026:
            df = pd.read_csv(file, dtype={'match_id': str})
            dfs.append(df)

df = pd.concat(dfs, ignore_index=True)

In [20]:
main = clean_teams(df)

In [21]:
main

,stat,home_value,away_value,match_id,date
0,general_Possession %,68.40,31.60,6afff6a7,Saturday March 7\nYankee Stadium
1,general_Shots,14.00,4.00,6afff6a7,Saturday March 7\nYankee Stadium
2,general_Shots on Target,6.00,1.00,6afff6a7,Saturday March 7\nYankee Stadium
3,general_Blocked Shots,4.00,2.00,6afff6a7,Saturday March 7\nYankee Stadium
4,general_Total Passes,648.00,255.00,6afff6a7,Saturday March 7\nYankee Stadium
...,...,...,...,...,...
2547,possession_81_85,45.45,54.55,7631e105,Sunday February 22\nLumen Field
2548,possession_86_90,29.17,70.83,7631e105,Sunday February 22\nLumen Field
2549,xg_Total Team XG,1.40,0.60,7631e105,Sunday February 22\nLumen Field
2550,xg_Shots,14.00,7.00,7631e105,Sunday February 22\nLumen Field


In [22]:
df = main.copy()

month_map = {
        "january": "01", "february": "02", "march": "03", "april": "04",
        "may": "05", "june": "06", "july": "07", "august": "08",
        "september": "09", "october": "10", "november": "11", "december": "12"
    }

days = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']


df['date'] = df['date'].str.split('\n').str[0]

df['date'] = df['date'].apply(lambda x: ' '.join([word for word in x.split() if word not in days]))


df['date'] = df['date'].apply(
        lambda x: re.sub(
            r"([A-Za-z]+)",
            lambda m: month_map[m.group(1).lower()],
            x
        )
    )

df['date'] = df['date'].apply(lambda x: x + " 2026" if re.search(r"\d{4}$", x) is None else x)

df['date'] = pd.to_datetime(df['date'], format="%m %d %Y")


In [23]:
df.to_csv('G:/My Drive/GitHubProjects/MLS/data/interim/matches/teams/team_stats_cleaned0326_post_26.csv', index=False)